# EDA Part 3 — Analyse temporelle

**Objectif :** Analyser les patterns temporels : volume par heure/jour, pics d'activité, saisonnalité. Créer des graphiques de séries temporelles. Identifier les fenêtres d'activité potentiellement suspectes.

**Datasets :**
- CIC-IDS-2017 → pas de timestamp explicite mais `Flow Duration` utilisable
- UNSW-NB15 → colonnes `Stime` et `Ltime` (timestamps Unix)
- Cybersecurity Threat Detection Logs → colonne `timestamp` (date complète)

## 0. Imports & Chargement

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

print('Libraries loaded ✅')

In [ ]:
# ⚠️ Modifie les chemins si nécessaire
PATH_CICIDS = '../Data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'
PATH_UNSW   = '../Data/UNSW-NB15_1.csv'
PATH_LOGS   = '../Data/cybersecurity_threat_detection_logs.csv'

unsw_columns = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur',
    'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service',
    'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb',
    'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
    'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt',
    'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl',
    'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src',
    'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label'
]

df_cic  = pd.read_csv(PATH_CICIDS, low_memory=False)
df_cic.columns = df_cic.columns.str.strip()

df_unsw = pd.read_csv(PATH_UNSW, names=unsw_columns, low_memory=False)

df_logs = pd.read_csv(PATH_LOGS, low_memory=False)

print('Datasets chargés ✅')

---
## 1. CIC-IDS-2017

> Ce fichier ne contient pas de timestamp absolu — il capture un vendredi après-midi uniquement. On utilise `Flow Duration` pour analyser la distribution temporelle des flux.

### 1.1 Distribution de la durée des flux

In [ ]:
# Suppression de la valeur aberrante -1
df_cic_clean = df_cic[df_cic['Flow Duration'] >= 0].copy()

# Conversion en secondes
df_cic_clean['Flow Duration (s)'] = df_cic_clean['Flow Duration'] / 1_000_000

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Tous les flux
df_cic_clean['Flow Duration (s)'].clip(upper=df_cic_clean['Flow Duration (s)'].quantile(0.99)).hist(
    bins=50, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('CICIDS2017 — Distribution durée des flux')
axes[0].set_xlabel('Durée (secondes)')
axes[0].set_ylabel('Nombre de flux')

# Par classe
for label, group in df_cic_clean.groupby('Label'):
    group['Flow Duration (s)'].clip(upper=60).hist(
        bins=50, ax=axes[1], alpha=0.6, label=label, edgecolor='white'
    )
axes[1].set_title('CICIDS2017 — Durée des flux par classe')
axes[1].set_xlabel('Durée (secondes)')
axes[1].set_ylabel('Nombre de flux')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.2 Identification des fenêtres suspectes

In [ ]:
# Flux très courts (< 0.1s) — typiques d'un DDoS
flux_courts = df_cic_clean[df_cic_clean['Flow Duration (s)'] < 0.1]
print(f'Flux très courts (< 0.1s) : {len(flux_courts):,} ({len(flux_courts)/len(df_cic_clean)*100:.1f}%)')
print('Distribution des labels pour les flux courts :')
print(flux_courts['Label'].value_counts())

# Flux très longs (> 60s) — potentiellement suspects
flux_longs = df_cic_clean[df_cic_clean['Flow Duration (s)'] > 60]
print(f'\nFlux très longs (> 60s) : {len(flux_longs):,} ({len(flux_longs)/len(df_cic_clean)*100:.1f}%)')
print('Distribution des labels pour les flux longs :')
print(flux_longs['Label'].value_counts())

---
## 2. UNSW-NB15

> Ce dataset contient `Stime` (timestamp Unix de début) et `Ltime` (timestamp Unix de fin). On les convertit en datetime pour analyser les patterns temporels.

### 2.1 Conversion des timestamps

In [ ]:
# Conversion timestamp Unix → datetime
df_unsw['datetime'] = pd.to_datetime(df_unsw['Stime'], unit='s', errors='coerce')
df_unsw = df_unsw.dropna(subset=['datetime'])

# Extraction heure et jour
df_unsw['heure']      = df_unsw['datetime'].dt.hour
df_unsw['jour']       = df_unsw['datetime'].dt.date
df_unsw['jour_semaine'] = df_unsw['datetime'].dt.day_name()

print(f'Période couverte : {df_unsw["datetime"].min()} → {df_unsw["datetime"].max()}')
print(f'Nombre de jours distincts : {df_unsw["jour"].nunique()}')

### 2.2 Volume par heure

In [ ]:
volume_heure = df_unsw.groupby('heure').size()

fig, ax = plt.subplots()
volume_heure.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('UNSW-NB15 — Volume de connexions par heure')
ax.set_xlabel('Heure')
ax.set_ylabel('Nombre de connexions')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### 2.3 Volume par jour

In [ ]:
volume_jour = df_unsw.groupby('jour').size()

fig, ax = plt.subplots()
volume_jour.plot(kind='line', ax=ax, color='steelblue', marker='o')
ax.set_title('UNSW-NB15 — Série temporelle du volume de connexions par jour')
ax.set_xlabel('Date')
ax.set_ylabel('Nombre de connexions')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 2.4 Fenêtres suspectes — attaques par heure

In [ ]:
# Volume d'attaques (Label=1) vs normal (Label=0) par heure
attaques_heure = df_unsw.groupby(['heure', 'Label']).size().unstack(fill_value=0)
attaques_heure.columns = ['Normal', 'Attaque']

fig, ax = plt.subplots()
attaques_heure.plot(kind='bar', ax=ax, color=['steelblue', 'salmon'])
ax.set_title('UNSW-NB15 — Attaques vs trafic normal par heure')
ax.set_xlabel('Heure')
ax.set_ylabel('Nombre de connexions')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# Heures avec le plus d'attaques
print('\nTop 5 des heures avec le plus d\'attaques :')
print(attaques_heure['Attaque'].sort_values(ascending=False).head())

---
## 3. Cybersecurity Threat Detection Logs

> Ce dataset est le plus riche temporellement — il couvre toute l'année 2024 avec un timestamp complet par événement.

### 3.1 Conversion des timestamps

In [ ]:
df_logs['datetime'] = pd.to_datetime(df_logs['timestamp'], errors='coerce')
df_logs = df_logs.dropna(subset=['datetime'])

df_logs['heure']        = df_logs['datetime'].dt.hour
df_logs['jour']         = df_logs['datetime'].dt.date
df_logs['mois']         = df_logs['datetime'].dt.month
df_logs['jour_semaine'] = df_logs['datetime'].dt.day_name()

print(f'Période couverte : {df_logs["datetime"].min()} → {df_logs["datetime"].max()}')

### 3.2 Volume par heure

In [ ]:
volume_heure_logs = df_logs.groupby('heure').size()

fig, ax = plt.subplots()
volume_heure_logs.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Logs — Volume d\'événements par heure')
ax.set_xlabel('Heure')
ax.set_ylabel('Nombre d\'événements')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### 3.3 Volume par mois — saisonnalité

In [ ]:
volume_mois = df_logs.groupby('mois').size()\n
fig, ax = plt.subplots()
volume_mois.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Logs — Volume d\'événements par mois (saisonnalité 2024)')
ax.set_xlabel('Mois')
ax.set_ylabel('Nombre d\'événements')
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc'], rotation=45)
plt.tight_layout()
plt.show()

### 3.4 Série temporelle par jour

In [ ]:
volume_jour_logs = df_logs.groupby('jour').size()

fig, ax = plt.subplots(figsize=(14, 4))
volume_jour_logs.plot(kind='line', ax=ax, color='steelblue', linewidth=0.8)
ax.set_title('Logs — Série temporelle du volume d\'événements par jour (2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Nombre d\'événements')
plt.tight_layout()
plt.show()

### 3.5 Fenêtres suspectes — événements malveillants par heure

In [ ]:
# Distribution des classes par heure
classes_heure = df_logs.groupby(['heure', 'threat_label']).size().unstack(fill_value=0)

fig, ax = plt.subplots()
classes_heure.plot(kind='bar', ax=ax, color=['steelblue', 'salmon', 'orange'])
ax.set_title('Logs — Distribution des menaces par heure')
ax.set_xlabel('Heure')
ax.set_ylabel('Nombre d\'événements')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# Heures avec le plus d'événements malveillants
if 'malicious' in classes_heure.columns:
    print('\nTop 5 des heures avec le plus d\'événements malveillants :')
    print(classes_heure['malicious'].sort_values(ascending=False).head())

### 3.6 Volume par jour de la semaine

In [ ]:
ordre_jours = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
volume_semaine = df_logs.groupby('jour_semaine').size().reindex(ordre_jours)

fig, ax = plt.subplots()
volume_semaine.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Logs — Volume d\'événements par jour de la semaine')
ax.set_xlabel('Jour')
ax.set_ylabel('Nombre d\'événements')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 3.7 Pics d'activité suspecte — top jours malveillants

In [ ]:
# Top 10 des jours avec le plus d'événements malveillants
malveillants_jour = df_logs[df_logs['threat_label'] == 'malicious'].groupby('jour').size()

print('Top 10 des jours avec le plus d\'événements malveillants :')
print(malveillants_jour.sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(14, 4))
malveillants_jour.plot(kind='line', ax=ax, color='salmon', linewidth=0.8)
ax.set_title('Logs — Série temporelle des événements malveillants par jour (2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Nombre d\'événements malveillants')
plt.tight_layout()
plt.show()

---
## 4. Observations & Prochaines étapes

> **À compléter après exécution du notebook**

### Observations principales
- CICIDS2017 : pas de timestamp absolu, analyse basée sur la durée des flux uniquement
- UNSW-NB15 : à compléter après exécution (heures de pic, jours suspects)
- Logs : à compléter après exécution (saisonnalité, heures suspectes, pics)

### Prochaines étapes
- Nettoyage des données (suppression outliers, valeurs aberrantes)
- Feature engineering temporel (créer des features heure/jour pour le modèle)
- Modélisation